# Prompt engineering: il singolo messaggio

Il codice del capitolo [«Prompt engineering: il singolo messaggio»](https://book.paithon.it/main/IngegneriaLLM/prompt-engineering.html), *Paithon Book*.

Le celle sono quelle del libro, nell'ordine in cui compaiono: il testo che le spiega sta nelle pagine, qui c'è solo la parte da eseguire e da rompere.

Generato da `scripts/genera-notebook.py`: le correzioni vanno fatte nelle pagine del libro, non qui.


> **Verificato il 2026-07-25** con torch 2.13.0, numpy 2.4.6, pandas 3.0.5, scikit-learn 1.9.0, transformers 5.14.1, diffusers 0.39.0, librosa 0.11.0, torch-geometric 2.8.0.post1. Tutte le celle di questo notebook sono state eseguite senza errori con quelle versioni; le librerie si muovono, e se qualcosa qui non gira piu' e' un errore del libro: [segnalalo](https://github.com/paithon-it/paithonbook/issues).


In [ ]:
# Su Colab quasi tutto c'è già; questa riga serve altrove.
%pip install -q numpy

In [ ]:
# Mostra il valore di ogni riga, come i commenti «# ->» del libro.
try:
    from IPython.core.interactiveshell import InteractiveShell
    InteractiveShell.ast_node_interactivity = 'all'
except ImportError:      # fuori da IPython non serve e non c'è
    pass

## Prompt engineering: il singolo messaggio

[Leggi la pagina](https://book.paithon.it/main/IngegneriaLLM/prompt-engineering.html)


### Le due manopole del campionamento


In [ ]:
import numpy as np

# Cinquanta candidati con logit estratti a caso una volta sola: il seme li
# tiene fissi per tutte le prove.
logit = np.random.default_rng(0).normal(scale=2, size=50)

def nucleo(logit, T, p=0.9):
    """Quanti token entrano nel nucleo top_p, a temperatura T."""
    prob = np.exp(logit / T - (logit / T).max())
    prob = np.sort(prob / prob.sum())[::-1]        # dal piu' probabile in giu'
    return int(np.searchsorted(np.cumsum(prob), p) + 1)   # i primi che arrivano a p

for T in (0.2, 1, 3):
    print(f"T = {T}: nucleo di {nucleo(logit, T)} token")

### Molte teste sono meglio di una: self-consistency


In [ ]:
from collections import Counter

def voto_di_maggioranza(risposte):
    """Data una lista di risposte finali campionate, restituisce la piu'
    frequente. A parita' di voti vince quella incontrata per prima, cosi'
    il risultato e' deterministico (Counter conserva l'ordine d'inserimento)."""
    conteggio = Counter(risposte)
    risposta, voti = conteggio.most_common(1)[0]
    return risposta, voti, len(risposte)

# Cinque catene di ragionamento indipendenti sulla stessa domanda
# ("54 km in 3 ore: quanti km in un'ora?"): di ognuna teniamo solo la
# risposta finale, perche' i passaggi sono stati scartati.
campioni = ["18", "18", "21", "18", "22"]

risposta, voti, totale = voto_di_maggioranza(campioni)
print(f"Risposta scelta: {risposta} ({voti}/{totale} voti)")
# -> Risposta scelta: 18 (3/5 voti)

## Loop engineering: progettare il ciclo

[Leggi la pagina](https://book.paithon.it/main/IngegneriaLLM/loop-engineering.html)


### Il cancello di verifica: verificare, non sperare


In [ ]:
# Un loop generate -> verify -> refine. Il generatore e' un finto LLM;
# il verificatore e' reale: e' il "cancello" (validation gate) del ciclo.

def verifica(slug):
    """Il gate: ritorna (ok, motivo). Nessun 'quasi': o passa o no."""
    if slug != slug.lower():
        return False, "deve essere tutto minuscolo"
    if " " in slug:
        return False, "niente spazi: usa il trattino"
    if len(slug) > 20:
        return False, f"troppo lungo ({len(slug)} > 20 caratteri)"
    return True, "ok"


# Finto LLM: legge l'ULTIMO motivo di rifiuto e corregge quello, come farebbe
# un modello a cui si passa il feedback. In un sistema vero qui c'e' il modello.
def genera(richiesta, feedback):
    if not feedback:                                  # primo tentativo, a freddo
        return "Guida Introduttiva a PyTorch"
    ultimo = feedback[-1]
    if "minuscolo" in ultimo:
        return "guida introduttiva a pytorch"
    if "spazi" in ultimo:
        return "guida-introduttiva-a-pytorch-per-tutti"
    if "lungo" in ultimo:
        return "guida-pytorch"
    return "guida-pytorch"


def loop(richiesta, max_tentativi=5):
    feedback = []  # la memoria del loop: cresce a ogni riflessione
    for i in range(1, max_tentativi + 1):
        candidata = genera(richiesta, feedback)      # execute
        ok, motivo = verifica(candidata)             # verify (il gate)
        print(f"tentativo {i}: {candidata!r} -> {motivo}")
        if ok:
            return candidata
        feedback.append(motivo)                      # reflect: annota l'errore
    raise RuntimeError(f"gate non superato in {max_tentativi} tentativi")


risultato = loop("crea uno slug per una guida a PyTorch")
print("accettato:", risultato)